# Phase 7A — HuggingFace Transformers

**Theory:** Attention mechanism, BERT (Bidirectional Encoder), GPT (decoder-only), word embeddings vs contextual embeddings.

**Install:** `pip install transformers torch datasets`

**Key concept:** Pre-trained transformer models are trained on billions of text tokens. You can use them directly with `pipeline()` or fine-tune them on your task with minimal data.

---

In [ ]:
TRANSFORMERS_AVAILABLE = False
try:
    from transformers import pipeline, AutoTokenizer, AutoModel
    import torch

    TRANSFORMERS_AVAILABLE = True
    print("transformers library ready")
except ImportError:
    print("Install: pip install transformers torch")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

---
## 1. HuggingFace Pipelines — Zero-Shot Use

The `pipeline()` function gives you state-of-the-art NLP in 2 lines. No training required.

In [ ]:
if TRANSFORMERS_AVAILABLE:
    # 1. Sentiment Analysis
    sentiment = pipeline("sentiment-analysis")

    texts = [
        "This product is absolutely amazing, I love it!",
        "Terrible experience, would not recommend.",
        "It's okay, nothing special but works fine.",
    ]
    results = sentiment(texts)
    print("=== Sentiment Analysis ===")
    for text, result in zip(texts, results):
        print(f"  '{text[:45]}...'")
        print(f"   → {result['label']} (confidence: {result['score']:.3f})")

In [ ]:
if TRANSFORMERS_AVAILABLE:
    # 2. Named Entity Recognition
    ner = pipeline("ner", grouped_entities=True)

    text = "Elon Musk founded SpaceX in Hawthorne, California in 2002."
    entities = ner(text)

    print("=== Named Entity Recognition ===")
    print(f"Text: {text}")
    for ent in entities:
        print(f"  '{ent['word']}' → {ent['entity_group']} (score: {ent['score']:.3f})")

In [ ]:
if TRANSFORMERS_AVAILABLE:
    # 3. Text Summarization
    summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

    article = """
    Machine learning is a method of data analysis that automates analytical model building.
    Based on the idea that systems can learn from data, identify patterns and make decisions
    with minimal human intervention, machine learning has become one of the most powerful
    technologies driving the fourth industrial revolution. The field encompasses supervised
    learning, unsupervised learning, and reinforcement learning, each with distinct algorithms
    and applications across industries including healthcare, finance, and transportation.
    """
    summary = summarizer(article, max_length=60, min_length=20, do_sample=False)
    print("=== Text Summarization ===")
    print(f"Summary: {summary[0]['summary_text']}")

In [ ]:
if TRANSFORMERS_AVAILABLE:
    # 4. Question Answering
    qa = pipeline("question-answering")

    context = """
    PyTorch is an open-source machine learning library developed by Facebook's AI Research lab.
    It provides a flexible deep learning framework, featuring dynamic computation graphs.
    PyTorch was released in 2016 and has become one of the most widely used frameworks
    for deep learning research and production.
    """
    questions = [
        "Who developed PyTorch?",
        "When was PyTorch released?",
    ]
    print("=== Question Answering ===")
    for q in questions:
        answer = qa(question=q, context=context)
        print(f"  Q: {q}")
        print(f"  A: '{answer['answer']}' (confidence: {answer['score']:.3f})")

---
## 2. Tokenization Deep Dive

Transformer models use subword tokenization (BPE/WordPiece) — not word-level.

In [ ]:
if TRANSFORMERS_AVAILABLE:
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

    texts = [
        "Machine learning is amazing.",
        "Tokenization splits words into subword units.",
    ]

    for text in texts:
        tokens = tokenizer.tokenize(text)
        ids = tokenizer.encode(text)
        print(f"Text   : {text}")
        print(f"Tokens : {tokens}")
        print(f"IDs    : {ids}")
        print(f"[CLS] = {tokenizer.cls_token}, [SEP] = {tokenizer.sep_token}")
        print()

---
## 3. Getting Sentence Embeddings

BERT produces contextual embeddings — the same word has different vectors depending on context. Use `[CLS]` token embedding as the sentence representation.

In [ ]:
if TRANSFORMERS_AVAILABLE:
    from transformers import AutoTokenizer, AutoModel
    import torch

    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    model = AutoModel.from_pretrained("bert-base-uncased")

    sentences = [
        "I love machine learning",
        "I hate machine learning",
        "Deep learning is a subset of machine learning",
        "The cat sat on the mat",
    ]

    # Get embeddings
    model.eval()
    embeddings = []

    with torch.no_grad():
        for sent in sentences:
            inputs = tokenizer(sent, return_tensors="pt", truncation=True, padding=True)
            outputs = model(**inputs)
            cls_embedding = outputs.last_hidden_state[:, 0, :]  # [CLS] token
            embeddings.append(cls_embedding.squeeze().numpy())

    embeddings = np.array(embeddings)
    print(f"Embedding shape: {embeddings.shape}  (4 sentences × 768 dimensions)")

    # Cosine similarity between sentences
    from sklearn.metrics.pairwise import cosine_similarity

    sim_matrix = cosine_similarity(embeddings)

    fig, ax = plt.subplots(figsize=(7, 5))
    labels = [s[:30] for s in sentences]
    sns.heatmap(
        sim_matrix,
        annot=True,
        fmt=".3f",
        cmap="YlOrRd",
        xticklabels=labels,
        yticklabels=labels,
        ax=ax,
    )
    ax.set_title("BERT Sentence Similarity (Cosine)")
    plt.xticks(rotation=30, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    print(
        "\nNote: 'I love ML' and 'I hate ML' have high similarity (similar structure)"
    )
    print("      'The cat sat on the mat' is dissimilar to all ML sentences")

---
## Summary — HuggingFace Ecosystem

| Resource | What It Is |
|----------|------------|
| `transformers` library | Load/run pretrained models |
| `pipeline()` | Zero-shot NLP (no training) |
| `AutoTokenizer` | Tokenize text for any model |
| `AutoModel` | Load any base model for embeddings |
| `AutoModelForSequenceClassification` | Fine-tune for text classification |
| `datasets` library | Load standard NLP benchmarks |
| `Trainer` API | Fine-tune any model on your data |

**Key models:**
- `bert-base-uncased` — encoder, good for classification/NER
- `distilbert-base-uncased` — faster, smaller BERT (97% performance)
- `gpt2` — decoder, good for text generation
- `facebook/bart-large-cnn` — encoder-decoder, good for summarization
- `sentence-transformers/all-MiniLM-L6-v2` — fast semantic similarity